# Titanic - Survival Prediction

**Goal:** Predict passenger survival
**Algorithm:** Random Forest + Feature Engineering
**Dataset:** [Titanic Competition](https://www.kaggle.com/c/titanic)

In [1]:
import kagglehub
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [1]:
import sys
if 'google.colab' in sys.modules:
    !pip install kagglehub -q
    from google.colab import files
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
else:
    print('Running locally')

Running locally - ready


## 1. Load Data

In [1]:
path = kagglehub.competition_download('titanic')
train = pd.read_csv(f'{path}/train.csv')
test = pd.read_csv(f'{path}/test.csv')
print('Train:', train.shape, 'Test:', test.shape)

## 2. Feature Engineering

In [1]:
def engineer(df):
    d = df.copy()
    d['Title'] = d['Name'].apply(lambda x: re.search(r' ([A-Za-z]+)\.', x).group(1)).map({'Mr':1,'Mrs':2,'Miss':3,'Master':4}).fillna(0).astype(int)
    d['FamilySize'] = d['SibSp'] + d['Parch'] + 1
    d['IsAlone'] = (d['FamilySize'] == 1).astype(int)
    d['Age'] = d['Age'].fillna(d['Age'].median())
    d['Fare'] = d['Fare'].fillna(d['Fare'].median())
    d['Embarked'] = d['Embarked'].fillna('S').map({'S':0,'C':1,'Q':2})
    d['Sex'] = d['Sex'].map({'male':0,'female':1})
    d['HasCabin'] = d['Cabin'].notna().astype(int)
    return d[['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked','Title','FamilySize','IsAlone','HasCabin']]

X = engineer(train)
y = train['Survived']
X_test = engineer(test)
print('Features:', list(X.columns))

## 3. Train

In [1]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
model.fit(X_tr, y_tr)
y_pred = model.predict(X_val)
print('Accuracy: %.4f' % accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred, target_names=['Died','Survived']))

Accuracy: 0.8324
              precision    recall  f1-score   support
        Died       0.84      0.90      0.87       110
    Survived       0.81      0.72      0.77        69
    accuracy                           0.83       179


## 4. Predict & Submit

In [1]:
sub = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': model.predict(X_test)})
sub.to_csv('titanic_submission.csv', index=False)
print('Predicted:', sub['Survived'].sum(), '/', len(sub))

Predicted: 168 / 418
